In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.1 MB/s eta 0:00:00


In [13]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV

In [3]:
df_train_traffic= pd.read_csv('https://raw.githubusercontent.com/Peter-korn/Group-project-ML/refs/heads/dataset-preprocessing/data/df_train_traffic_encoded_scaled.csv')

In [4]:
df_train_traffic.corr(numeric_only = True)

,Трафик,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час",Выручка,month_sin,month_cos,pca_1,pca_2,...,pca_4,Населенный пункт_fold_traffic,Регион_fold_traffic,"Дата открытия, категориальный_Новый","Дата открытия, категориальный_Открыт давно","Дата открытия, категориальный_Средний по возрасту","Торговая площадь, категориальный_Большой","Торговая площадь, категориальный_Маленький","Торговая площадь, категориальный_Очень большой","Торговая площадь, категориальный_Средний"
Трафик,1.000000,0.324248,0.368781,0.326461,-0.010776,0.870008,-0.062518,-0.095464,0.317763,0.277410,...,-0.010268,0.429912,0.394184,-0.115367,0.261361,-0.004424,0.336103,-0.215238,0.298473,-0.193516
Численность населения,0.324248,1.000000,0.359612,0.239939,0.007984,0.342601,0.000988,0.000376,0.313222,0.279992,...,-0.092087,0.639873,0.730869,-0.022104,0.180661,-0.058665,0.084963,-0.009099,0.136660,-0.105037
Количество домохозяйств,0.368781,0.359612,1.000000,0.343596,-0.162855,0.221596,0.002253,0.000455,0.695874,-0.133067,...,0.032038,0.400940,0.280116,0.004938,0.156104,-0.073880,0.155818,-0.015411,0.112657,-0.150341
"Трафик пеший, в час",0.326461,0.239939,0.343596,1.000000,0.054366,0.204025,0.002969,-0.000085,0.412315,0.172433,...,-0.029051,0.235573,0.195635,-0.069186,0.105243,0.020147,0.108389,-0.026546,0.115953,-0.103498
"Трафик авто, в час",-0.010776,0.007984,-0.162855,0.054366,1.000000,0.024917,-0.001535,0.000172,-0.057811,0.086684,...,-0.105795,0.011213,0.017330,0.027065,-0.024645,-0.015199,-0.010161,-0.042172,0.002578,0.041845
Выручка,0.870008,0.342601,0.221596,0.204025,0.024917,1.000000,-0.023646,0.004036,0.145866,0.288989,...,0.013658,0.455567,0.481349,-0.124200,0.307605,-0.016377,0.351671,-0.248273,0.370343,-0.201927
month_sin,-0.062518,0.000988,0.002253,0.002969,-0.001535,-0.023646,1.000000,-0.003347,0.003348,0.003287,...,0.004274,0.001303,0.002009,-0.020496,0.003061,0.018418,0.001086,-0.001056,0.001735,-0.000573
month_cos,-0.095464,0.000376,0.000455,-0.000085,0.000172,0.004036,-0.003347,1.000000,0.000314,0.000686,...,0.000769,0.000539,0.000548,-0.003496,0.000591,0.003111,-0.000179,0.000121,0.000339,-0.000059
pca_1,0.317763,0.313222,0.695874,0.412315,-0.057811,0.145866,0.003348,0.000314,1.000000,0.000098,...,0.000284,0.314167,0.245456,-0.065224,0.172090,-0.013271,0.144813,-0.018331,0.104318,-0.136339
pca_2,0.277410,0.279992,-0.133067,0.172433,0.086684,0.288989,0.003287,0.000686,0.000098,1.000000,...,0.003015,0.266555,0.277978,-0.075936,0.101649,0.028250,0.059963,-0.034953,0.096199,-0.050820


In [14]:
train_traffic = pd.read_csv('https://raw.githubusercontent.com/Peter-korn/Group-project-ML/refs/heads/main/data/df_train_traffic_encoded_scaled.csv')
test_traffic = pd.read_csv('https://raw.githubusercontent.com/Peter-korn/Group-project-ML/refs/heads/main/data/df_test_traffic_encoded_scaled.csv')
train_bill = pd.read_csv('https://raw.githubusercontent.com/Peter-korn/Group-project-ML/refs/heads/main/data/df_train_average_bill_encoded_scaled.csv')
test_bill = pd.read_csv('https://raw.githubusercontent.com/Peter-korn/Group-project-ML/refs/heads/main/data/df_test_average_bill_encoded_scaled.csv')

In [18]:
cols = [f'{s}_м{m}' for s in ['новый', 'средний', 'старый'] for m in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]]

In [25]:
def fnc(train, test, target):
  X, y = train.drop(columns= [target]), train[target]
  X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size = 0.2, random_state= 1337)

  param = {'depth': [4, 6, 8, 10], 'learning_rate': [0.03, 0.05, 0.1], 'l2_leaf_reg': [1, 3, 5, 7], 'iterations': [300, 500, 800]}
  srch = RandomizedSearchCV(CatBoostRegressor(random_state = 1337, verbose = 0), param_distributions= param, n_iter = 15, cv = 3, scoring = 'r2', random_state = 1337, n_jobs = -1).fit(X_tr, y_tr)
  kaif = srch.best_estimator_
  print(srch.best_params_)

  print(mean_absolute_error(y_val, kaif.predict(X_val)), r2_score(y_val, kaif.predict(X_val)))
  n = len(kaif.predict(test[X.columns])) // 36
  final = pd.DataFrame(kaif.predict(test[X.columns]).reshape(n, 36), columns = cols)
  final.index.name = 'магазин'
  return kaif, final

In [26]:
kaif_t, final_t = fnc(train_traffic, test_traffic, 'Трафик')

{'learning_rate': 0.05, 'l2_leaf_reg': 1, 'iterations': 500, 'depth': 10}
5314.795657953928 0.7279058238650105


In [28]:
kaif_b, final_b = fnc(train_bill, test_bill, 'Средний чек')

{'learning_rate': 0.05, 'l2_leaf_reg': 1, 'iterations': 500, 'depth': 10}
107.9662259020491 0.7896462604275307


In [29]:
final_t

,новый_м1,новый_м2,новый_м3,новый_м4,новый_м5,новый_м6,новый_м7,новый_м8,новый_м9,новый_м10,...,старый_м3,старый_м4,старый_м5,старый_м6,старый_м7,старый_м8,старый_м9,старый_м10,старый_м11,старый_м12
магазин,,,,,,,,,,,,,,,,,,,,,
0,57384.164104,58193.328198,61937.842873,61986.304984,61949.262378,60627.793720,61454.043123,62171.100796,63426.401799,63649.873661,...,67813.128436,67932.508902,68409.938684,66109.622259,66385.364099,66700.190078,67869.494353,68502.875425,66547.187748,67179.342048
1,63776.108493,64672.388998,68645.534970,68708.243282,68869.405962,67664.478700,68214.446385,68737.437936,69929.140627,70422.572227,...,75120.984356,75267.895514,76041.610688,73994.021575,74062.913484,74217.123526,75387.359968,76043.459234,74039.051652,74719.792086
2,70110.119488,71095.794950,75470.815098,75780.239684,75971.652526,74579.423178,75059.376619,75287.476406,76466.198223,77144.790474,...,81942.858636,82349.434949,83090.502032,80898.357692,80858.209479,81111.449728,82205.837947,83018.652123,81033.270601,81781.013607
3,75514.234460,76175.125629,79257.784175,80802.910443,81549.157513,80768.585673,81597.047845,81902.268523,82852.896500,83111.029431,...,89500.133225,90322.462280,91909.673937,90421.599807,90599.913367,89998.648244,90716.186138,91315.853147,89428.009102,90116.542086
4,53537.741518,54482.922814,57916.503175,58176.958334,58557.394462,57282.414481,57957.207747,58044.068915,59061.844557,59363.272972,...,62049.459052,62365.663882,62957.631487,61331.655414,61557.482915,61303.770737,62193.688650,62303.320006,60580.652218,61191.569601
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324,61272.168529,61705.858222,64848.255873,65696.579099,66674.972309,66426.119973,67239.879009,66843.834211,66860.792584,67033.499209,...,68169.053912,68883.952764,69846.631008,69523.484121,69848.642976,69317.048460,69476.819247,69417.630997,67778.008329,68469.399770
325,57229.567826,57666.803691,60638.488219,61441.616483,62388.373662,62125.636856,62819.812258,62381.688612,62351.043808,62461.333259,...,62210.584988,62917.683490,63939.354983,63457.611123,63788.168072,63049.212174,63269.837122,63046.699218,61421.847388,62087.919975
326,57229.567826,57666.803691,60638.488219,61441.616483,62388.373662,62125.636856,62819.812258,62381.688612,62351.043808,62461.333259,...,62210.584988,62917.683490,63939.354983,63457.611123,63788.168072,63049.212174,63269.837122,63046.699218,61421.847388,62087.919975


In [30]:
final_b

,новый_м1,новый_м2,новый_м3,новый_м4,новый_м5,новый_м6,новый_м7,новый_м8,новый_м9,новый_м10,...,старый_м3,старый_м4,старый_м5,старый_м6,старый_м7,старый_м8,старый_м9,старый_м10,старый_м11,старый_м12
магазин,,,,,,,,,,,,,,,,,,,,,
0,952.042430,977.240444,1015.406577,963.757477,943.713475,903.485955,878.641069,895.746576,926.385455,986.475214,...,1219.973240,1162.162736,1146.457537,1100.052769,1063.679586,1072.607873,1100.142513,1161.692508,1220.310336,1351.719257
1,1121.796026,1148.064383,1190.302579,1135.056007,1113.699270,1069.651233,1042.566829,1060.971526,1091.257370,1160.312065,...,1422.027274,1353.790334,1336.772401,1286.678644,1249.523258,1259.750736,1286.925517,1362.973939,1422.991724,1584.281625
2,1360.973315,1387.196200,1434.620774,1374.391214,1350.376556,1300.881762,1273.870974,1293.873354,1325.235191,1405.834712,...,1685.020400,1614.290103,1594.691118,1535.142414,1495.986017,1507.734308,1535.800601,1624.342842,1685.967633,1897.050423
3,1434.678637,1471.909508,1509.345459,1463.623515,1439.971367,1395.487112,1373.769276,1383.905271,1394.745093,1498.490248,...,1831.810404,1778.892881,1752.397685,1694.790769,1661.013666,1672.018029,1679.222030,1788.811927,1847.716644,2040.635319
4,859.333951,878.734000,917.309632,864.433048,846.962483,808.990386,785.810687,797.730817,823.927856,882.712519,...,1114.915758,1064.005954,1048.049366,1001.721433,969.777941,979.728612,1003.713414,1067.033907,1121.967920,1243.076848
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324,922.907851,936.283458,965.421485,943.339354,947.206240,930.685696,935.976416,931.208909,921.874158,963.959075,...,1053.812179,1028.794886,1028.336020,1009.068274,1012.757158,1012.124668,1002.553820,1044.434439,1070.730130,1192.107634
325,822.789493,835.994625,864.421922,842.127823,848.272583,832.205982,837.127404,829.978255,820.163504,862.923691,...,952.416947,929.249804,931.068812,912.846255,916.690372,913.676239,903.941451,941.521307,967.566219,1070.107965
326,822.789493,835.994625,864.421922,842.127823,848.272583,832.205982,837.127404,829.978255,820.163504,862.923691,...,952.416947,929.249804,931.068812,912.846255,916.690372,913.676239,903.941451,941.521307,967.566219,1070.107965
